In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from collections import Counter

# Download NLTK resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print("✅ All libraries ready!")

✅ All libraries ready!


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/anupagarwal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/anupagarwal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anupagarwal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/anupagarwal/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/anupagarwal/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


# Load the IMDB Dataset


In [3]:
# Load the dataset
# Source: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

df = pd.read_csv('imdb_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (50000, 2)
Columns: ['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [5]:
# Let's look at a few sample reviews
for i, row in df.head(3).iterrows():
    print(f"\n{'='*60}")
    print(f"Sentiment: {row['sentiment'].upper()}")
    print(f"Review: {row['review'][:200]}...")


Sentiment: POSITIVE
Review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Sentiment: POSITIVE
Review: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece...

Sentiment: POSITIVE
Review: I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is wi...


# PART 1: Text → Tokens
## Tokenization & Normalization Choices


## 1.1 What is a Token?
### A token is a single unit of text. Think of it as "words" but more flexible.

In [6]:
# Simple example: What is a token?
sentence = "I love NLP!"

# METHOD 1: Simplest approach - split by spaces
tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
print("Number of tokens:", len(tokens_simple))

Split by space: ['I', 'love', 'NLP!']
Number of tokens: 3


In [7]:
# But wait... what about punctuation?
sentence = "Hello, world! How are you?"

tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
# Problem: "Hello," and "you?" have punctuation attached!

Split by space: ['Hello,', 'world!', 'How', 'are', 'you?']


# 1.2 Tokenization Methods Compared


In [8]:
from nltk.tokenize import word_tokenize, TreebankWordTokenizer

# A complex sentence to tokenize
text = "I can't believe http://wow.com is FREE!!! It's amazing @user 🔥"

print("Original text:")
print(text)
print("\n" + "="*60)

Original text:
I can't believe http://wow.com is FREE!!! It's amazing @user 🔥



In [9]:
# Method 1: Simple whitespace split
tokens_whitespace = text.split()
print("\n1️⃣ Whitespace split:")
print(tokens_whitespace)
print(f"   → {len(tokens_whitespace)} tokens")


1️⃣ Whitespace split:
['I', "can't", 'believe', 'http://wow.com', 'is', 'FREE!!!', "It's", 'amazing', '@user', '🔥']
   → 10 tokens


In [10]:
# Method 2: NLTK's word_tokenize (smarter!)
tokens_nltk = word_tokenize(text)
print("\n2️⃣ NLTK word_tokenize:")
print(tokens_nltk)
print(f"   → {len(tokens_nltk)} tokens")
print("   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)")


2️⃣ NLTK word_tokenize:
['I', 'ca', "n't", 'believe', 'http', ':', '//wow.com', 'is', 'FREE', '!', '!', '!', 'It', "'s", 'amazing', '@', 'user', '🔥']
   → 18 tokens
   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)


In [11]:
# Method 3: Regex-based tokenization (custom control)
tokens_regex = re.findall(r'\b\w+\b', text)
print("\n3️⃣ Regex (\\b\\w+\\b):")
print(tokens_regex)
print(f"   → {len(tokens_regex)} tokens")
print("   → Notice: Punctuation and URLs handled differently!")


3️⃣ Regex (\b\w+\b):
['I', 'can', 't', 'believe', 'http', 'wow', 'com', 'is', 'FREE', 'It', 's', 'amazing', 'user']
   → 13 tokens
   → Notice: Punctuation and URLs handled differently!


In [12]:
# COMPARISON TABLE
print("\n" + "="*60)
print("COMPARISON: Same text, different tokenizers")
print("="*60)
comparison = pd.DataFrame({
    'Method': ['Whitespace', 'NLTK', 'Regex'],
    'Token Count': [len(tokens_whitespace), len(tokens_nltk), len(tokens_regex)],
    'Handles Contractions': ['❌', '✅', '❌'],
    'Handles Punctuation': ['❌', '✅', '✅'],
})
comparison


COMPARISON: Same text, different tokenizers


,Method,Token Count,Handles Contractions,Handles Punctuation
0,Whitespace,10,❌,❌
1,NLTK,18,✅,✅
2,Regex,13,❌,✅
